# Analisi statistica sulla popolazione dei runner

Questo notebook documenta e lancia il job Spark per l'analisi della popolazione dei runner.

Eseguire il seguente comando una sola volta e poi riavviare il kernel.

In [ ]:
%pip install matplotlib

Definisco le variabili necessarie (nella reale esecuzione queste variabili vengono importate da un file di configurazione esterno).

In [ ]:
import os

CLICKHOUSE_URL = os.getenv("CLICKHOUSE_JDBC_URL", "jdbc:clickhouse://clickhouse:8123/bigintensive")
CLICKHOUSE_PROPS = {
    "user": os.getenv("CLICKHOUSE_USER", "default"),
    "password": os.getenv("CLICKHOUSE_PASSWORD", ""),
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
}
CLICKHOUSE_TABLE = os.getenv("CLICKHOUSE_TABLE", "running_samples")


POSTGRES_URL = os.getenv("POSTGRES_JDBC_URL", "jdbc:postgresql://postgres:5432/bigintensive")
POSTGRES_PROPS = {
    "user": os.getenv("POSTGRES_USER", "postgres"),
    "password": os.getenv("POSTGRES_PASSWORD", "postgres"),
    "driver": "org.postgresql.Driver",
}
POSTGRES_TABLE = os.getenv("POSTGRES_TABLE", "anthropometric_values")


KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:19092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC", "heart-rate-events")
KAFKA_STARTING_OFFSETS = os.getenv("SPARK_STREAM_STARTING_OFFSETS", "latest")

print("Variabili di ambiente importare")

Importo tutte le librerie necessarie e configuro l'ambiente Spark per eseguire l'analisi dei dati.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("pyspark").setLevel(logging.ERROR)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg,  lag, countDistinct, first, row_number, to_date, radians, sin, cos, sqrt, atan2
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

print("Librerie importate")

Rimuovo eventuali executor pod rimasti orfani da esecuzioni precedenti fallite, prima di creare una nuova sessione.

In [ ]:
import json
import ssl
import urllib.request

SA_DIR = "/var/run/secrets/kubernetes.io/serviceaccount"
NAMESPACE = "bigintensive"


def _k8s_request(path, method="GET"):
    with open(f"{SA_DIR}/token") as f:
        token = f.read()
    ctx = ssl.create_default_context(cafile=f"{SA_DIR}/ca.crt")
    req = urllib.request.Request(
        f"https://kubernetes.default{path}",
        method=method,
        headers={"Authorization": f"Bearer {token}"},
    )
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.load(resp)


def pulisci_executor_orfani():
    pods = _k8s_request(
        f"/api/v1/namespaces/{NAMESPACE}/pods?labelSelector=spark-role%3Dexecutor"
    )["items"]
    attivi = 0
    for pod in pods:
        nome = pod["metadata"]["name"]
        fase = pod["status"]["phase"]
        if fase in ("Failed", "Succeeded"):
            _k8s_request(f"/api/v1/namespaces/{NAMESPACE}/pods/{nome}", method="DELETE")
            print(f"eliminato pod {nome} ({fase})")
        else:
            attivi += 1

    # Le conf-map degli executor non vengono rimosse se il driver muore: bloccano il riavvio con 409.
    if attivi == 0:
        cms = _k8s_request(f"/api/v1/namespaces/{NAMESPACE}/configmaps")["items"]
        for cm in cms:
            nome = cm["metadata"]["name"]
            if nome.startswith("spark-exec-") and nome.endswith("-conf-map"):
                _k8s_request(
                    f"/api/v1/namespaces/{NAMESPACE}/configmaps/{nome}", method="DELETE"
                )
                print(f"eliminata configmap {nome}")
    else:
        print(f"{attivi} executor ancora attivi: configmap non toccate")

    print("pulizia completata")


pulisci_executor_orfani()
print("Executor orfani eliminati")

Creazione di una sessione Spark con le configurazioni appropriate per l'analisi dei dati dei runner.

In [ ]:
# In client mode la JVM del driver parte prima del builder: la memoria va fissata qui.
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 2g pyspark-shell"

# Senza questo, gli executor non hanno ownerReference e restano orfani se il driver muore.
DRIVER_POD_NAME = os.environ["SPARK_DRIVER_POD_NAME"]

# Numero di executor: cambialo e riavvia il kernel per confrontare i tempi di esecuzione.
NUM_EXECUTORS = 4

# Un getOrCreate fallito lascia un SparkContext zombie che va chiuso prima di riprovare.
from pyspark import SparkContext

if SparkContext._active_spark_context is not None:
    try:
        SparkContext._active_spark_context.stop()
    except Exception as errore:
        print(f"contesto precedente non chiudibile: {errore}")

spark = (
    SparkSession.builder.appName("running-population-analysis")
    .config("spark.log.level", "ERROR")
    .config("spark.master", "k8s://https://kubernetes.default:443")
    .config("spark.kubernetes.namespace", "bigintensive")
    .config("spark.kubernetes.authenticate.driver.mounted", "true")
    .config("spark.kubernetes.authenticate.driver.serviceAccountName", "spark")
    .config("spark.kubernetes.authenticate.executor.mounted", "true")
    .config("spark.kubernetes.driver.pod.name", DRIVER_POD_NAME)
    .config("spark.kubernetes.executor.deleteOnTermination", "true")
    .config("spark.driver.host", "jupyter.bigintensive.svc.cluster.local")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "7078")
    .config("spark.driver.blockManager.port", "7079")
    .config("spark.kubernetes.container.image", "apache/spark:3.5.3")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "2g")
    .config("spark.kubernetes.executor.request.cores", "400m")
    .config("spark.kubernetes.executor.limit.cores", "1")
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.executor.instances", str(NUM_EXECUTORS))
    .config("spark.sql.shuffle.partitions", str(NUM_EXECUTORS * 4))
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.3,"
        "com.clickhouse:clickhouse-jdbc:0.6.3,"
        "com.clickhouse:clickhouse-http-client:0.6.3,"
        "org.apache.httpcomponents.client5:httpclient5:5.3.1",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession creata")

Importiamo le tabelle dei dati dei runner da clickhouse e Postgresql.

In [ ]:
# Fisso a 4 e non legato a NUM_EXECUTORS: il lavoro da svolgere deve restare identico nei due test.
NUM_PARTIZIONI = 4
ATLETA_MIN = 1
ATLETA_MAX = 10000

df = (
    spark.read.format("jdbc")
    .option("url", CLICKHOUSE_URL)
    .option("dbtable", CLICKHOUSE_TABLE)
    .option("user", CLICKHOUSE_PROPS["user"])
    .option("password", CLICKHOUSE_PROPS["password"])
    .option("driver", CLICKHOUSE_PROPS["driver"])
    .option("partitionColumn", "athlete_id")
    .option("lowerBound", ATLETA_MIN)
    .option("upperBound", ATLETA_MAX)
    .option("numPartitions", NUM_PARTIZIONI)
    .load()
)

# Tabella piccola: partizionarla aggiungerebbe overhead senza guadagno.
df_postgres = (
    spark.read.format("jdbc")
    .option("url", POSTGRES_URL)
    .option("dbtable", POSTGRES_TABLE)
    .option("user", POSTGRES_PROPS["user"])
    .option("password", POSTGRES_PROPS["password"])
    .option("driver", POSTGRES_PROPS["driver"])
    .load()
)

print("partizioni clickhouse:", df.rdd.getNumPartitions())
print("partizioni postgres:  ", df_postgres.rdd.getNumPartitions())

df.show(5)
df_postgres.show(5)

Creazione delle finestre temporali per l'analisi dei dati dei runner, utilizzando le funzioni di finestra di Spark.

In [ ]:
finestra_temporale = Window.partitionBy("athlete_id", "session_id").orderBy("sample_id")
finestra_temporale_5min = finestra_temporale.rowsBetween(-60, 0)

print("Finestre temporali create")

Pulisco il dataset da valori nulli e duplicati.

In [ ]:
df_ordinato = df.orderBy(col("athlete_id"), col("session_id"), col("sample_id"))
df_ordinato.show(5)

df_pulito = df_ordinato.dropDuplicates(["athlete_id", "session_id", "sample_id"])
df_pulito.show(5)

df_pulito_null = df_pulito.filter(
    col("athlete_id").isNotNull()
    & col("session_id").isNotNull()
    & col("sample_id").isNotNull()
    & col("heart_rate").isNotNull()
    & col("latitude").isNotNull()
    & col("longitude").isNotNull()
    & col("timestamp").isNotNull()
)
df_pulito_null.show(5)

Aggiungo e calcolo la colonna BMI (Body Mass Index) nel dataset dei runner, utilizzando i dati di peso e altezza.

In [ ]:
df_postgres_aggiornato = df_postgres.withColumn("BMI", col("peso_kg") / (col("altezza_cm") * col("altezza_cm")/10000))
df_postgres_aggiornato.show(5)

Pulisco il dataset di Postgres selezionando solo le colonne necessarie per l'analisi, dopo aver calcolato il BMI.

In [ ]:
df_postgres_ridotto = df_postgres_aggiornato.select("athlete_id", "data_rilevazione", "peso_kg", "altezza_cm", "BMI")
df_postgres_ridotto.show(5)

Calcolo della velocità media tramite la posizione GPS tramite le formule trigonometriche distribuite e della deriva cardiaca puntuale e percentuale per ogni atleta e sessione, usando le finestre appena create.

In [ ]:
R=6371000  # Raggio della Terra in metri
df_deriva_cardiaca = df_pulito_null \
    .withColumn("lat_prec", lag("latitude", 1).over(finestra_temporale)) \
    .withColumn("lon_prec", lag("longitude", 1).over(finestra_temporale)) \
    .withColumn("lat_rad", radians(col("latitude"))) \
    .withColumn("lon_rad", radians(col("longitude"))) \
    .withColumn("lat_prec_rad", radians(col("lat_prec"))) \
    .withColumn("lon_prec_rad", radians(col("lon_prec"))) \
    .withColumn("dlat", col("lat_rad") - col("lat_prec_rad")) \
    .withColumn("dlon", col("lon_rad") - col("lon_prec_rad")) \
    .withColumn("a", sin(col("dlat") / 2) ** 2 + cos(col("lat_rad")) * cos(col("lat_prec_rad")) * sin(col("dlon") / 2) ** 2) \
    .withColumn("c", 2 * atan2(sqrt(col("a")), sqrt(1 - col("a")))) \
    .withColumn("distanza", R * col("c")) \
    .withColumn("velocita_puntuale", col("distanza") / 5) \
    .withColumn("velocita_media", avg("velocita_puntuale").over(finestra_temporale_5min)) \
    .withColumn("frequenza_cardiaca_media", avg("heart_rate").over(finestra_temporale_5min)) \
    .withColumn("Efficienza_puntuale", col("velocita_puntuale") / col("frequenza_cardiaca_media")) \
    .withColumn("Efficienza_puntuale_iniziale", first("Efficienza_puntuale", ignorenulls=True).over(finestra_temporale)) \
    .withColumn("Deriva_cardiaca_percentuale", (col("Efficienza_puntuale")- col("Efficienza_puntuale_iniziale")) / col("Efficienza_puntuale_iniziale") * 100)

df_deriva_cardiaca.show(5)

Trovo la velocità che causa la deriva cardiaca più alta per ogni atleta e sessione, filtrando i dati per valori di deriva cardiaca superiori al 5%. Ne estrae il primo punto di crisi per ogni sessione.

In [ ]:
finestra_sessione_tempo = Window.partitionBy("athlete_id", "session_id").orderBy("timestamp")
df_crisi_ordinate = df_deriva_cardiaca \
                    .filter(col("Deriva_cardiaca_percentuale") > 5) \
                    .withColumn("riga_crisi", row_number().over(finestra_sessione_tempo))
                    
primo_punto_di_crisi_per_sessione = df_crisi_ordinate \
        .filter(col("riga_crisi") == 1) \
        .select("timestamp", "velocita_media", "Deriva_cardiaca_percentuale", "athlete_id", "session_id")

primo_punto_di_crisi_per_sessione.show(5)

Calcolo il numero di corse per ogni atleta e unisco questo dato al dataset dei punti di crisi.

In [ ]:
df_conteggio_corse = primo_punto_di_crisi_per_sessione.groupBy("athlete_id").agg(countDistinct("session_id").alias("numero_corse"))

df_conteggio_corse.show(5)

Eseguo il join con il dataset di Postgres e delle corse per ottenere un dataset finale con tutte le informazioni necessarie per l'analisi della popolazione dei runner.

In [ ]:
crisi = primo_punto_di_crisi_per_sessione.alias("crisi")
antropometria = df_postgres_ridotto.alias("antropometria")

# Associa la misura piu recente disponibile prima della data della crisi.
crisi_con_antropometria = crisi.join(
    antropometria,
    (col("crisi.athlete_id") == col("antropometria.athlete_id"))
    & (col("antropometria.data_rilevazione") <= to_date(col("crisi.timestamp"))),
    how="inner",
)

finestra_antropometria = Window.partitionBy(
    "crisi.athlete_id", "crisi.session_id"
).orderBy(col("antropometria.data_rilevazione").desc())

primo_punto_di_crisi_per_sessione = (
    crisi_con_antropometria
    .withColumn("riga_antropometria", row_number().over(finestra_antropometria))
    .filter(col("riga_antropometria") == 1)
    .select(
        col("crisi.timestamp").alias("timestamp"),
        col("crisi.velocita_media").alias("velocita_media"),
        col("crisi.Deriva_cardiaca_percentuale").alias("Deriva_cardiaca_percentuale"),
        col("crisi.athlete_id").alias("athlete_id"),
        col("crisi.session_id").alias("session_id"),
        col("antropometria.peso_kg").alias("peso_kg"),
        col("antropometria.altezza_cm").alias("altezza_cm"),
        col("antropometria.BMI").alias("BMI"),
    )
)

df_preanalisi = primo_punto_di_crisi_per_sessione.join(df_conteggio_corse, ["athlete_id"], how="left")

df_preanalisi.show(5)

Calcolo la matrice di correlazione tra le variabili del dataset finale per identificare eventuali relazioni tra le caratteristiche dei runner e i punti di crisi.

In [ ]:
colonne_da_analizzare = ["peso_kg", "altezza_cm", "BMI", "velocita_media", "numero_corse"]

df_ml = df_preanalisi.select(colonne_da_analizzare).na.drop()
numero_righe_ml = df_ml.limit(2).count()

if numero_righe_ml < 2:
    print(
        "Correlazione non calcolata: servono almeno due righe complete "
        f"e ne sono disponibili {numero_righe_ml}."
    )
else:
    assembler = VectorAssembler(inputCols=colonne_da_analizzare, outputCol="features")
    df_ml = assembler.transform(df_ml)

    matrice_correlazione = Correlation.corr(df_ml, "features", "pearson").head()[0]
    print("Matrice di correlazione:")

Visualizzo la matrice di correlazione come heatmap: il triangolo superiore e mascherato perche la matrice e simmetrica, e il colore va dal blu (correlazione negativa) al rosso (positiva).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

if numero_righe_ml < 2:
    print("Heatmap non disegnata: matrice di correlazione non disponibile.")
else:
    matrice = matrice_correlazione.toArray()
    etichette = ["Peso (kg)", "Altezza (cm)", "BMI", "Velocita media", "Numero corse"]

    # La matrice e simmetrica: mostro solo il triangolo inferiore per non ripetere gli stessi valori
    maschera = np.triu(np.ones_like(matrice, dtype=bool), k=1)
    matrice_mascherata = np.ma.masked_array(matrice, mask=maschera)

    mappa_colori = plt.get_cmap("RdBu_r").copy()
    mappa_colori.set_bad(color="white")

    fig, ax = plt.subplots(figsize=(8, 6.5), dpi=120)
    immagine = ax.imshow(
        matrice_mascherata,
        cmap=mappa_colori,
        norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1),
    )

    ax.set_xticks(np.arange(len(etichette)))
    ax.set_yticks(np.arange(len(etichette)))
    ax.set_xticklabels(etichette, rotation=35, ha="right", fontsize=10)
    ax.set_yticklabels(etichette, fontsize=10)

    # Griglia sottile tra le celle
    ax.set_xticks(np.arange(len(etichette) + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(etichette) + 1) - 0.5, minor=True)
    ax.grid(which="minor", color="white", linewidth=2)
    ax.tick_params(which="minor", length=0)
    for lato in ax.spines.values():
        lato.set_visible(False)

    for riga in range(matrice.shape[0]):
        for colonna in range(riga + 1):
            valore = matrice[riga, colonna]
            # Testo bianco sulle celle sature, nero su quelle chiare
            colore_testo = "white" if abs(valore) > 0.55 else "#1a1a1a"
            ax.text(
                colonna, riga, f"{valore:.2f}",
                ha="center", va="center",
                color=colore_testo, fontsize=11,
                fontweight="bold" if riga != colonna else "normal",
            )

    barra = fig.colorbar(immagine, ax=ax, shrink=0.82, pad=0.03)
    barra.set_label("Coefficiente di Pearson", fontsize=10)
    barra.outline.set_visible(False)

    ax.set_title(
        "Matrice di correlazione\nCaratteristiche antropometriche e punto di crisi",
        fontsize=13, fontweight="bold", pad=16,
    )

    fig.tight_layout()
    plt.show()

    # Coppie piu correlate, escludendo la diagonale
    coppie = [
        (etichette[i], etichette[j], matrice[i, j])
        for i in range(len(etichette))
        for j in range(i)
    ]
    coppie.sort(key=lambda voce: abs(voce[2]), reverse=True)

    print("Correlazioni piu forti:")
    for prima, seconda, valore in coppie[:5]:
        print(f"  {prima:<15} - {seconda:<15} {valore:+.3f}")

## Confronto dei tempi di esecuzione

Misuro la durata della pipeline completa al variare del numero di executor. Cambia `NUM_EXECUTORS` nella cella della sessione Spark, riavvia il kernel e riesegui il notebook per ottenere il tempo con una configurazione diversa.

In [ ]:
import time

# count() forza l'esecuzione dell'intero DAG: show(5) si fermerebbe alle prime partizioni.
if numero_righe_ml < 2:
    righe = 0
    durata = 0
    executor_attivi = 0
    print("Conteggio righe non eseguito: servono almeno due righe complete.")
else:

    inizio = time.perf_counter()
    righe = df_preanalisi.count()
    durata = time.perf_counter() - inizio
    # L'API Java restituisce anche il driver; lo escludiamo dal conteggio.
    executor_attivi = max(0, spark.sparkContext._jsc.sc().getExecutorMemoryStatus().size() - 1)

print(f"executor richiesti: {NUM_EXECUTORS}")
print(f"executor attivi:    {executor_attivi}")
print(f"righe elaborate:    {righe:,}")
print(f"tempo:              {durata:.2f} s")

## Chiudi la sessione Spark

Esegui questa cella solo quando hai concluso tutte le analisi nel notebook.

In [ ]:
spark.stop()
print("SparkSession closed.")